# SOOP 韩语转中文 SRT 字幕与硬字幕视频生成

本 Notebook 使用 Google Drive 存储模型缓存、输出文件以及任务恢复状态。

### 💡 推荐的 CPU -> GPU 工作流：
1. **首先在 CPU 运行时启动**（不启用 GPU，以节省 Colab 计算额度）。
2. 运行 **挂载谷歌网盘**、**克隆/更新代码库**、**安装依赖** 的单元格。
3. 运行 **Prepare 阶段（准备）**。它会下载低画质视频并提取音频，随后会**自动停止运行**并给出提示。
4. 点击顶部菜单栏的 **代码执行程序 (Runtime) -> 更改运行时类型 (Change runtime type)**，选择 **T4 GPU** 并保存。（这会重连并重启运行时）。
5. 再次运行所有单元格。管道（Pipeline）会自动**跳过**已完成的下载和音频提取步骤（从网盘缓存读取），直接在 GPU 运行时下运行 ASR 语音识别，生成 `<vod-id>.ko.srt`。
6. 下载 `<vod-id>.ko.srt`，在本地使用 `translate-srt` 命令翻译成 `<vod-id>.zh.srt`，然后再上传回网盘输出目录中。
7. 在输出目录下创建并上传分段 CSV 文件，手动运行最后的 **Render 阶段（渲染）** 单元格来生成带有中文硬字幕的剪辑视频。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 如果此 Notebook 尚未处于项目目录中，请设置 REPO_URL 并运行此单元格。
REPO_URL = 'https://github.com/digiprospector/kr-sc-srt.git'  # 示例: 'https://github.com/your-name/kr-sc-srt.git'
PROJECT_DIR = '/content/kr-sc-srt'

import os
import pathlib
import subprocess
project_path = pathlib.Path(PROJECT_DIR)
if REPO_URL and not project_path.exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
elif REPO_URL and (project_path / '.git').exists():
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=True)
if project_path.exists():
    %cd {PROJECT_DIR}
else:
    print('请设置 REPO_URL，或从项目目录中上传/打开此 Notebook。')

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!python -m pip install -q -U pip
!python -m pip install -q -r requirements.txt yt-dlp

In [ ]:
import subprocess
import sys

ROOT = '/content/drive/MyDrive/kr-sc-srt'
MODEL_CACHE = f'{ROOT}/models'
LAST_JOB = f'{ROOT}/last_job.json'

# 首次运行：粘贴 VOD URL。后续重新运行可将 VOD_URL 保持为空，并启用 RESUME_LAST。
VOD_URL = ''
RESUME_LAST = True
ASR_CHUNK_S = 180  # ASR 语音识别音频分片时长（秒）。如果 Colab 报内存不足（OOM）可以调小。
TEST = False  # 设置为 True 则仅处理前 10 分钟音频，用于快速测试。

os.makedirs(ROOT, exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

if not VOD_URL and not os.path.exists(LAST_JOB):
    VOD_URL = input('首次运行请输入 VOD URL: ').strip()
    if not VOD_URL:
        raise ValueError('首次运行必须设置 VOD_URL。')

def run_cli(cmd):
    print(' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, cmd)

In [ ]:
cmd = [sys.executable, '-u', '-m', 'kr_sc_srt', 'prepare', '--root', ROOT, '--model-cache-dir', MODEL_CACHE]
if VOD_URL:
    cmd.append(VOD_URL)
elif RESUME_LAST:
    if not os.path.exists(LAST_JOB):
        raise ValueError(f'未在 {LAST_JOB} 找到保存的任务状态。首次运行请设置 VOD_URL。')
    cmd.append('--resume-last')
else:
    raise ValueError('请设置 VOD_URL 或启用 RESUME_LAST')

if ASR_CHUNK_S:
    cmd.extend(['--asr-chunk-s', str(ASR_CHUNK_S)])

if TEST:
    cmd.append('--test')

run_cli(cmd)

raise SystemExit('\n=== PREPARE 阶段已完成 ===\n\n1. 下载韩语字幕并将其翻译为 <vod-id>.zh.srt\n2. 将 <vod-id>.zh.srt 和你的分段 CSV 文件上传回 Google Drive\n3. 手动运行下方的 Render 单元格。')

在 prepare 阶段完成后，下载 `<vod-id>.ko.srt` 并在你本地的机器上进行翻译：

```bash
python -m kr_sc_srt translate-srt <vod-id>.ko.srt <vod-id>.zh.srt --api-key-env OPENAI_API_KEY
```

将翻译好的 `<vod-id>.zh.srt` 上传/保存回同一个任务输出目录下。然后在该目录下创建或修改分段 CSV 文件。格式如下：

```csv
name,start,end
part-01,00:01:30,00:05:00
part-02,00:05:00,00:08:30.500
```

默认情况下，`render --resume-last` 会在保存的输出目录中寻找 `<job-name>.csv`。


In [ ]:
cmd = [sys.executable, '-u', '-m', 'kr_sc_srt', 'render', '--root', ROOT, '--resume-last']
if not os.path.exists(LAST_JOB):
    raise ValueError(f'在 {LAST_JOB} 未找到保存的任务。请先运行 prepare 阶段。')
if TEST:
    cmd.append('--test')

run_cli(cmd)

if os.path.exists(LAST_JOB):
    os.remove(LAST_JOB)
    print('渲染成功完成。已删除 last_job.json 以清理任务状态。')